# Chapter 1: Causal Inference for Data Science — Code Examples & Extended Simulations

This Jupyter Notebook provides fully executable Python code, extended simulations, and practical industry applications accompanying **Chapter 1: Introducing Causality** from *Causal Inference for Data Science* (2025) by Aleix Ruiz de Villa.

---

## Table of Contents
1. **Simulation 1: Spurious Correlation & Common Cause (Confounder)**
2. **Simulation 2: Observational Bias vs. A/B Testing (RCT)**
3. **Simulation 3: Website Traffic & Seasonality (Time-Series Confounding)**
4. **Simulation 4: Empirical vs. Data-Generating Distributions (Glivenko-Cantelli Theorem)**


In [ ]:
# Environment Setup & Core Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for exact reproducibility
np.random.seed(1234)

# Set global plotting aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Setup completed successfully. Ready for simulations.")



---
## 1. Simulation: Spurious Correlation & Common Confounder

As discussed in Section 1.4.1 of the textbook, two completely unrelated variables—such as arcade revenue and computer science doctorates awarded—can exhibit extremely high statistical correlation ($r > 0.98$). This occurs when both variables are driven by a shared underlying confounder (e.g., total population growth over time).

In this simulation, we generate synthetic data where:
- $\text{Population} \sim \mathcal{N}(\mu(t), \sigma^2)$
- $\text{Revenue} = f(\text{Population}) + \epsilon_1$
- $\text{Doctorates} = g(\text{Population}) + \epsilon_2$

Notice that $\text{Revenue}$ and $\text{Doctorates}$ are generated **independently** conditional on $\text{Population}$.


In [ ]:
# Simulation 1: Spurious Correlation (Listing 1.2 extended)

# Define time horizon (2000 to 2009)
years = np.arange(2000, 2010)

# Simulate common cause: Population growth in millions
# Base population of 280 million + 3 million annual growth + random noise
population = 280 + 3 * (years - 2000) + np.random.normal(loc=0, scale=0.1, size=len(years))

# Simulate arcade revenue (in billions of USD) purely as a function of population
revenue = 1.25 + (population - 280) * 0.015 + np.random.normal(loc=0, scale=0.05, size=len(years))

# Simulate CS doctorates awarded purely as a function of population
doctorates = 700 + (population - 280) * 30 + np.random.normal(loc=0, scale=10, size=len(years))

# Create DataFrame
df_spurious = pd.DataFrame({
    'Year': years,
    'Population_Millions': population,
    'Arcade_Revenue_Billions': revenue,
    'CS_Doctorates': doctorates
})

# Calculate raw Pearson correlation between doctorates and revenue
raw_corr = np.corrcoef(df_spurious['CS_Doctorates'], df_spurious['Arcade_Revenue_Billions'])[0, 1]
print(f"Raw Pearson Correlation between CS Doctorates and Arcade Revenue: {raw_corr:.4f}")

# Visualizing Spurious Correlation vs Common Cause
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:red'
ax1.set_xlabel('Year')
ax1.set_ylabel('Arcade Revenue ($ Billions)', color=color)
ax1.plot(df_spurious['Year'], df_spurious['Arcade_Revenue_Billions'], color=color, marker='o', linewidth=2, label='Arcade Revenue')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('CS Doctorates Awarded', color=color)
ax2.plot(df_spurious['Year'], df_spurious['CS_Doctorates'], color=color, marker='s', linestyle='--', linewidth=2, label='CS Doctorates')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'Spurious Correlation (r = {raw_corr:.2f}) Driven by Common Confounder (Population)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()



---
## 2. Simulation: Observational Bias vs. A/B Testing (RCT)

In Section 1.4.2, the textbook introduces a medical scenario where a doctor evaluates a new drug. 
- **Observational Data**: The doctor predominantly gives the new drug to older, higher-risk patients. Because age negatively impacts recovery ($Age \rightarrow Recovery$), a simple comparison of recovery rates between treated and untreated groups shows that the new drug performs *worse* than the old treatment, despite the new drug actually having a positive treatment effect!
- **Experimental Data (A/B Test / RCT)**: Treatment is assigned completely at random ($P(Treatment) = 0.5$), severing the link between age and treatment ($Age \not\rightarrow Treatment$), isolating the true Average Treatment Effect (ATE).


In [ ]:
# Simulation 2: Observational Confounding vs. RCT Assignment

n_patients = 5000

# Generate Patient Ages (20 to 80 years old)
age = np.random.uniform(20, 80, size=n_patients)

# Define Confounded Treatment Assignment (Observational)
# Older patients are significantly more likely to receive Treatment = 1
prob_treatment_obs = 1 / (1 + np.exp(-(age - 50) / 10))  # Logistic probability curve
treatment_obs = np.random.binomial(1, prob_treatment_obs)

# Define True Potential Outcomes
# True causal effect of treatment is +0.15 (15% improvement in recovery rate)
true_ate = 0.15
baseline_recovery_prob = 0.90 - 0.008 * age

# Individual Recovery Probabilities
prob_recovery_untreated = np.clip(baseline_recovery_prob, 0, 1)
prob_recovery_treated = np.clip(baseline_recovery_prob + true_ate, 0, 1)

# Realized Outcome for Observational Data
recovery_obs = np.where(treatment_obs == 1, 
                        np.random.binomial(1, prob_recovery_treated), 
                        np.random.binomial(1, prob_recovery_untreated))

df_obs = pd.DataFrame({'Age': age, 'Treatment': treatment_obs, 'Recovery': recovery_obs})

# Naive ATE Calculation in Observational Data
naive_ate_obs = df_obs[df_obs['Treatment'] == 1]['Recovery'].mean() - df_obs[df_obs['Treatment'] == 0]['Recovery'].mean()

print("=== OBSERVATIONAL DATA ANALYSIS ===")
print(f"Mean Recovery Rate (Treated):   {df_obs[df_obs['Treatment'] == 1]['Recovery'].mean():.4f}")
print(f"Mean Recovery Rate (Untreated): {df_obs[df_obs['Treatment'] == 0]['Recovery'].mean():.4f}")
print(f"Naive Observational ATE:        {naive_ate_obs:.4f}  (TRUE ATE = {true_ate})")
print("--> Result is severely biased downward because older/sicker patients received the drug!
")

# --- NOW SIMULATE RANDOMIZED CONTROLLED TRIAL (A/B TEST) ---
# Random assignment completely decoupled from Age
treatment_rct = np.random.binomial(1, 0.5, size=n_patients)

recovery_rct = np.where(treatment_rct == 1, 
                       np.random.binomial(1, prob_recovery_treated), 
                       np.random.binomial(1, prob_recovery_untreated))

df_rct = pd.DataFrame({'Age': age, 'Treatment': treatment_rct, 'Recovery': recovery_rct})

rct_ate = df_rct[df_rct['Treatment'] == 1]['Recovery'].mean() - df_rct[df_rct['Treatment'] == 0]['Recovery'].mean()

print("=== RANDOMIZED CONTROLLED TRIAL (A/B TEST) ANALYSIS ===")
print(f"Mean Recovery Rate (Treated):   {df_rct[df_rct['Treatment'] == 1]['Recovery'].mean():.4f}")
print(f"Mean Recovery Rate (Untreated): {df_rct[df_rct['Treatment'] == 0]['Recovery'].mean():.4f}")
print(f"Experimental RCT ATE:          {rct_ate:.4f}  (TRUE ATE = {true_ate})")
print("--> Randomization eliminates confounding bias and uncovers the true causal effect!")



---
## 3. Simulation: Website Redesign & Seasonality (Time-Series Confounding)

In Section 1.3.1, the textbook presents a motivating e-commerce example. An e-commerce company launches a website redesign right after a major holiday. Visits rise after launch, but was it due to the redesign or post-holiday recovery?


In [ ]:
# Simulation 3: E-Commerce Web Visits with Seasonality Noise

days = np.arange(-30, 31) # Day 0 is Launch Date

# Underlying Seasonality Trend: Holiday slump around Day -10, rebound around Day +10
seasonality = 15 - 8 * np.exp(-((days + 5)**2) / 30) + 0.3 * days

# True Web Redesign Effect: Increases daily traffic by +5k visits starting Day 0
true_redesign_effect = np.where(days >= 0, 5.0, 0.0)

# Observed Visits (in thousands) with Gaussian daily fluctuations
noise = np.random.normal(0, 1.5, size=len(days))
visits = seasonality + true_redesign_effect + noise

df_web = pd.DataFrame({'Day': days, 'Visits_Thousands': visits, 'Post_Launch': days >= 0})

# Visualizing Pre vs Post Launch
plt.figure(figsize=(11, 5))
plt.plot(df_web[df_web['Day'] < 0]['Day'], df_web[df_web['Day'] < 0]['Visits_Thousands'], 
         color='teal', linestyle='--', marker='o', label='Old Website Version')
plt.plot(df_web[df_web['Day'] >= 0]['Day'], df_web[df_web['Day'] >= 0]['Visits_Thousands'], 
         color='crimson', marker='o', label='New Website Version')
plt.axvline(x=0, color='black', linestyle=':', label='Website Relaunch Date (Day 0)')

plt.title('E-Commerce Web Traffic Pre/Post Launch (Confounded by Seasonality)', fontsize=13, fontweight='bold')
plt.xlabel('Days Relative to Launch')
plt.ylabel('Daily Visits (Thousands)')
plt.legend()
plt.tight_layout()
plt.show()



---
## 4. Simulation: Empirical vs. Data-Generating Distributions

Section 1.5.1 discusses the Glivenko-Cantelli Theorem: as sample size $n \rightarrow \infty$, the empirical distribution $\tilde{F}_n(x)$ converges uniformly to the true data-generating distribution $F(x)$.


In [ ]:
# Simulation 4: Convergence of Empirical Expectation to Data-Generating Expectation

true_p = 0.60 # True coin bias
sample_sizes = np.logspace(1, 4.5, num=50, dtype=int)

empirical_means = []
for n in sample_sizes:
    flips = np.random.binomial(1, true_p, size=n)
    empirical_means.append(np.mean(flips))

plt.figure(figsize=(10, 5))
plt.plot(sample_sizes, empirical_means, marker='o', color='darkblue', linewidth=1.5, label='Empirical Mean (x̄)')
plt.axhline(y=true_p, color='red', linestyle='--', linewidth=2, label=f'True Expectation E[X] = {true_p}')
plt.xscale('log')
plt.title('Glivenko-Cantelli Convergence: Sample Mean vs True Data-Generating Expectation', fontsize=13, fontweight='bold')
plt.xlabel('Sample Size (n) [Log Scale]')
plt.ylabel('Proportion / Expectation')
plt.legend()
plt.tight_layout()
plt.show()

